# 🔋 The Next-Gen Battery Race: Materials Discovery with Qiskit
## Official Hackathon Submission Notebook
### Team Name: [Enter Team Name]
### Team Members: [Member 1, Member 2, ...]

---

### 🏆 Challenge Roadmap & Scoring (100 Points Total):
* **Part 1 (The Training Ground):** Clean Hydrogen Fuel ($H_2$) Potential Energy Surface (10 pts)
* **Part 2 (The Industry Benchmark):** Lithium-Graphite ($Li-C$) Anode Baseline Voltage (20 pts)
* **Part 3 (Novel Materials Discovery):** Design & Simulate an Alternative Battery Material (45 pts)
* **Part 4 (Scientific Defense & Pitch):** Engineering Trade-Off Presentation (25 pts)


In [ ]:
# Optional: If running on Google Colab, uncomment and run this line once:
# !pip install qiskit>=1.2.0 qiskit-aer>=0.14.0 matplotlib numpy scipy

import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize

from qiskit.circuit.library import EfficientSU2
from qiskit.primitives import StatevectorEstimator

from utils.battery_helpers import (
    load_benchmarks,
    get_h2_hamiltonian,
    get_battery_hamiltonian,
    build_custom_battery_model,
    create_custom_hamiltonian,
    calculate_battery_voltage,
    evaluate_circuit_cost,
    plot_pes_comparison,
    plot_materials_showdown,
    check_part1,
    check_part2,
    check_part3,
    ELEMENT_DATABASE
)

benchmarks = load_benchmarks()
print("✅ Battery materials environment loaded successfully!")


---
## Part 1: The Training Ground — Molecular Hydrogen ($H_2$) PES (10 pts)
**Goal:** Sweep the bond distance $R \in [0.4, 2.5]$ Å, solve for the ground-state energy at each distance using VQE, and plot the potential energy curve.

**Instructions:**
1. Loop over `distances`.
2. For each distance `r`, retrieve the Hamiltonian using `get_h2_hamiltonian(r)`.
3. Set up the VQE loop and find the minimum energy.
4. Append each optimized energy to `vqe_h2_energies`.


In [ ]:
distances = benchmarks['H2_dissociation']['bond_lengths_angstrom']
fci_energies = benchmarks['H2_dissociation']['FCI_Hartree']
hf_energies = benchmarks['H2_dissociation']['Hartree_Fock_Hartree']

vqe_h2_energies = []

# ==========================================================
# TODO: YOUR CODE HERE (Part 1)
# Hint:
# estimator = StatevectorEstimator()
# ansatz_h2 = EfficientSU2(num_qubits=2, reps=1, entanglement='linear')
# initial_angles = np.random.uniform(0, 2 * np.pi, ansatz_h2.num_parameters)
#
# for r in distances:
#     H = get_h2_hamiltonian(r)
#     res = minimize(lambda p: float(estimator.run([(ansatz_h2, H, p)]).result()[0].data.evs),
#                    initial_angles, method='SLSQP', options={'maxiter': 50})
#     vqe_h2_energies.append(res.fun)
#     initial_angles = res.x  # Pro-tip: warm-start next distance with optimal angles!
# ==========================================================



### Part 1 Self-Check & Plot
Run the self-check below to verify your equilibrium bond length and chemical accuracy:


In [ ]:
# Uncomment and run once your loop is implemented:
# check_part1(distances, vqe_h2_energies, fci_energies)
# plot_pes_comparison(distances, vqe_h2_energies, fci_energies, hf_energies, title="Hydrogen Dissociation PES")


---
## Part 2: The Industry Benchmark — Lithium on Carbon ($Li-C$) (20 pts)
**Goal:** Compute the ground-state energy of the Lithium-Carbon electrode interface ($E(Li\text{-}C)$). 
Then, calculate the **Binding Energy** ($\Delta E_{\text{bind}}$) and the **Theoretical Battery Cell Voltage** ($V_{\text{cell}}$).

**The Physics:**
$$\Delta E_{\text{bind}} = E(Li\text{-}C) - \left[ E(C\text{-host}) + E(Li) \right]$$
$$V_{\text{cell}} = -\frac{\Delta E_{\text{bind}}}{e} \quad (\text{Industry Target: } \approx 3.42\text{ V})$$


In [ ]:
# Load the baseline 4-qubit Li-C Hamiltonian
H_lic = get_battery_hamiltonian("Li-C")
print("Li-C Battery Hamiltonian (4 Qubits):")
print(H_lic)

# ==========================================================
# TODO: YOUR CODE HERE (Part 2)
# 1. Define your ansatz for the 4-qubit Li-C system (e.g. EfficientSU2(4, reps=1))
# 2. Execute VQE to find the minimum energy E_lic
# 3. Store your optimized energy in e_lic_optimized
# ==========================================================

e_lic_optimized = None  # Replace with your optimized energy


### Part 2 Self-Check
Run the self-check below to verify your baseline battery voltage and store `baseline_results`:


In [ ]:
# Uncomment and run once you find e_lic_optimized:
# baseline_results = check_part2(e_lic_optimized)


---
## Part 3: Novel Materials Discovery & Custom Model Design (45 pts)
### *"Can You Build a Better Battery Model? Can You Prove It?"*

In Part 1 and Part 2, you mastered the computational pipeline:
1. **What to find:** The ground state energy $E_{\text{bound}}$ of an intercalated metal-electrode system.
2. **How to calculate properties:** Subtract isolated reference energies to get the binding energy ($\Delta E_{\text{bind}}$), then derive the open-circuit cell voltage ($V_{\text{cell}} = -\Delta E_{\text{bind}} / (z \cdot e)$).
3. **Why it matters:** The strength and stability of this atomic bond dictates voltage, safety, and cost.

Now it is your turn to **design your own battery model**! Move beyond standard Lithium-Graphite and discover what happens when you alter the chemistry.

### Choose Your Elements & Host:
Use `build_custom_battery_model(metal=..., host=..., distance_angstrom=None)`:
* **Metal Ion Choices:**
  * `'Li'`: Lithium (Valence 1, commercial standard)
  * `'Na'`: Sodium (Valence 1, table salt, abundant, 10x cheaper grid storage)
  * `'K'`:  Potassium (Valence 1, large ionic radius, fast diffusion kinetics)
  * `'Mg'`: Magnesium (Valence 2, multivalent carrier, high volumetric energy without dendrite fires)
  * `'Ca'`: Calcium (Valence 2, abundant multivalent alternative to Lithium)
  * `'Al'`: Aluminum (Valence 3, trivalent multi-electron transfer, non-flammable & ultra-safe)
  * `'Zn'`: Zinc (Valence 2, aqueous electrolyte compatible, recyclable, low toxicity)
* **Host Substrate Choices:**
  * `'Carbon'`: Standard graphite ($sp^2$ carbon lattice)
  * `'Silicon'`: High-capacity alloy host ($10\times$ theoretical capacity)
  * `'N-doped-Carbon'`: Polar nitrogen defects that trap ions securely and prevent fires
  * `'B-doped-Carbon'`: P-type boron defects that enhance electronic conductivity
  * `'Sulfur'`: High-energy multi-electron conversion cathode host
* **Intercalation Distance:** Leaving `distance_angstrom=None` automatically selects the relaxed equilibrium distance!

*(Alternatively, advanced teams can define a custom Hamiltonian from scratch using `create_custom_hamiltonian()`!)*


In [ ]:
# Build your team's custom battery model!
# Example: Try 'Na' on 'Carbon', or 'Mg' on 'Silicon', or 'Al' on 'N-doped-Carbon'
my_model = build_custom_battery_model(
    metal="Na",              # Choose: 'Li', 'Na', 'K', 'Mg', 'Ca', 'Al', 'Zn'
    host="Carbon",           # Choose: 'Carbon', 'Silicon', 'N-doped-Carbon', 'B-doped-Carbon', 'Sulfur'
    distance_angstrom=None   # None = automatically use relaxed equilibrium bond distance!
)

print(f"Created Model: {my_model['name']}")
print(f"Application Context: {my_model['application_tradeoff']}")
H_candidate = my_model["hamiltonian"]

# ==========================================================
# TODO: YOUR CODE HERE (Part 3)
# 1. Run VQE to find the ground state energy of your custom model
# 2. Store your optimized energy in e_candidate_optimized
# ==========================================================

e_candidate_optimized = None  # Replace with your optimized candidate energy


### Part 3 Discovery Check & Materials Showdown Plot
Verify your candidate's voltage and compare it side-by-side against the baseline:


In [ ]:
# Uncomment and run once you find e_candidate_optimized:
# candidate_results = check_part3(e_candidate_optimized, model_info=my_model)

# Plot the Materials Showdown comparison bar chart
# plot_materials_showdown([baseline_results, candidate_results])


---
## Part 4: The Scientific Defense & Presentation Pitch (25 pts)

*Answer the following questions to defend your discovered material to the judges:*

### 1. Which engineering dimension did your team prioritize, and why?
* [ ] **Dimension A: Maximum Voltage & Range** (Silicon Anode)
* [ ] **Dimension B: Low-Cost Grid Storage** (Sodium / Potassium)
* [ ] **Dimension C: Safety & Fire Prevention** (Nitrogen-Doped Carbon / Multivalent Mg/Al)

> **Defense:** [Explain why your team chose to solve this specific battery problem.]

### 2. The Materials Showdown Table
Fill in this comparison table using the numbers calculated from your quantum simulation:

| Metric | Baseline ($Li-C$ Graphite) | Your Candidate ([Enter Name]) | Real-World Advantage |
| :--- | :---: | :---: | :--- |
| **Active Metal Element** | Lithium ($Li$) | [e.g. Sodium ($Na$)] | [e.g. 10x cheaper raw material] |
| **Host Electrode** | $sp^2$ Carbon (Graphite) | [e.g. Carbon / Silicon] | [...] |
| **Binding Energy ($\Delta E_{\text{bind}}$)** | -3.42 eV | [...] eV | [...] |
| **Theoretical Cell Voltage** | 3.42 V | [...] V | [Higher voltage or grid-optimized?] |

### 3. Industry Pitch: Why should energy companies invest in your material?
> [Summarize your final recommendation based on your quantum simulation data.]

---
## Final Submission Checklist
Before submitting:
- [ ] Team name and member names filled in at the top.
- [ ] Part 1 executed with visible PES curve.
- [ ] Part 2 executed with baseline voltage ($≈ 3.42\text{ V}$).
- [ ] Part 3 executed with candidate voltage and showdown plot.
- [ ] Part 4 pitch and showdown table completed.
- [ ] Repository set to **Private** with judges invited as collaborators.
- [ ] Submit your repository link and presentation slides on the [Official Submission Form](https://forms.gle/qMA5vMGVBeEmbKMa9).
